# Leakage-Free Evaluation

In [1]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

sys.path.append('../..')  # repo root, so `scripts` is importable
from scripts.forecasting_helpers import (
    FEATURES, CATEGORICAL_FEATURES, NUMERIC_FEATURES, TARGETS,
    load_xgboost_hyperparameters, load_and_clean, build_model_inputs,
    encode_features, calculate_all_metrics,
)

np.random.seed(35)

In [2]:
FILE_PATH        = '../../data/preprocessed/final_nfi_ch2018_merged/NFI_with_Climate_Averages_RCP45.csv'
HYPERPARAMS_PATH = Path('../../data/hyperparameters/nfi1/best_params.json')

TARGET_LABELS = {'BASFPH_next_INVNR': 'Basal Area', 'HWSW_prop_next_INVNR': 'Broadleaf Fraction'}

XGB_PARAMS = load_xgboost_hyperparameters(HYPERPARAMS_PATH)

## 1  Build train / test split

In [3]:
df = load_and_clean(FILE_PATH)
df['DATUMF'] = pd.to_datetime(df['DATUMF'], format='%d.%m.%Y', errors='coerce')

train_df = df[df['INVNR'].isin([150, 250])].copy()
test_df  = df[df['INVNR'] == 450].copy()

# Verify no temporal overlap
last_train_target = df[df['INVNR'] == 350]['DATUMF'].max()   # NFI3 = training targets for NFI2 rows
first_test_feature = test_df['DATUMF'].min()
assert last_train_target < first_test_feature, 'Overlap detected!'

print(f'Last training target date  (NFI3): {last_train_target.date()}')
print(f'First test feature date    (NFI4): {first_test_feature.date()}')
print(f'Gap: {(first_test_feature - last_train_target).days} days')
print('Zero temporal leakage confirmed.')
print(f'\nTrain rows : {len(train_df)}')
print(f'Test rows  : {len(test_df)}')

Last training target date  (NFI3): 2005-10-27
First test feature date    (NFI4): 2012-04-06
Gap: 2353 days
Zero temporal leakage confirmed.

Train rows : 240
Test rows  : 120


## 2  Encode features

In [4]:
scope   = build_model_inputs(df, [150, 250])
X_train = scope['X_train']
y_train = scope['y_train']
X_test  = encode_features(test_df, scope['cats_map'], scope['final_features'])
y_test  = test_df[TARGETS]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

X_train: (240, 29)  |  X_test: (120, 29)


## 3  Train and predict

In [5]:
predictions = {}
for target in TARGETS:
    model = xgb.XGBRegressor(**XGB_PARAMS[target], importance_type='gain')
    model.fit(X_train, y_train[target])
    predictions[target] = model.predict(X_test)
    print(f'Trained for {target}')

Trained for BASFPH_next_INVNR
Trained for HWSW_prop_next_INVNR


## 4  Metrics

In [6]:
def compute_metrics(y_true, y_pred, label):
    """Display-formatted (rounded) metrics for the comparison table."""
    m = calculate_all_metrics(y_true, y_pred)
    return {
        'Model': label, 'N': m['N'],
        'RMSE':  round(m['RMSE'], 3),
        '%RMSE': round(m['%RMSE'], 1),
        'R2':    round(m['R_squared'], 3),
        'Bias':  round(m['Bias'], 3),
    }

# Reference from model_evaluation.ipynb
REF = {
    'BASFPH_next_INVNR':    {'Model': 'Reference (train NFI1-3, leaky)', 'N': 3915, 'RMSE': 10.482, '%RMSE': 32.1, 'R2': 0.658, 'Bias':  1.012},
    'HWSW_prop_next_INVNR': {'Model': 'Reference (train NFI1-3, leaky)', 'N': 3915, 'RMSE':  0.122, '%RMSE': 31.6, 'R2': 0.904, 'Bias':  0.004},
}

rows = []
for target in TARGETS:
    rows.append(REF[target] | {'Target': target})
    row = compute_metrics(y_test[target].values, predictions[target],
                          'Leakage-free (train NFI1-2 only)')
    row['Target'] = target
    rows.append(row)

results = pd.DataFrame(rows)[['Target', 'Model', 'N', 'RMSE', '%RMSE', 'R2', 'Bias']]
print(results.to_string(index=False))

              Target                            Model    N   RMSE  %RMSE    R2   Bias
   BASFPH_next_INVNR  Reference (train NFI1-3, leaky) 3915 10.482   32.1 0.658  1.012
   BASFPH_next_INVNR Leakage-free (train NFI1-2 only)  120  4.417   12.6 0.845 -1.200
HWSW_prop_next_INVNR  Reference (train NFI1-3, leaky) 3915  0.122   31.6 0.904  0.004
HWSW_prop_next_INVNR Leakage-free (train NFI1-2 only)  120  0.067   12.0 0.920 -0.014
